models.py

In [14]:
from pydantic import BaseModel, Field, model_validator


class BearerConfig(BaseModel):
    bearer_id: int = Field(ge=1, le=9)
    protocol: str | None = Field(default=None, pattern="^(tcp|udp)$")
    target_bps: int | None = None  # bits per second
    active: bool = False


class ThroughputStats(BaseModel):
    bearer_id: int
    ue_id: int
    bytes_tx: int = 0  # uplink (MS->SS)
    bytes_rx: int = 0  # downlink (SS->MS)
    start_ts: float | None = None
    last_update_ts: float | None = None
    protocol: str | None = None
    target_bps: int | None = None


class UEState(BaseModel):
    ue_id: int = Field(ge=1, le=100)
    bearers: dict[int, BearerConfig] = {}
    stats: dict[int, ThroughputStats] = {}

    @model_validator(mode="before")
    def init_defaults(cls, values):
        if values.get("bearers") is None:
            values["bearers"] = {}
        if values.get("stats") is None:
            values["stats"] = {}
        return values


# Request body schemas (REST API)
class AttachUERequest(BaseModel):
    ue_id: int = Field(ge=1, le=100)


class AddBearerRequest(BaseModel):
    bearer_id: int = Field(ge=1, le=9)


class StartTrafficRequest(BaseModel):
    protocol: str = Field(pattern="^(tcp|udp)$")
    Mbps: float | None = None
    kbps: float | None = None
    bps: float | None = None

    @model_validator(mode="after")
    def exactly_one_throughput(self):
        provided = [v for v in [self.Mbps, self.kbps, self.bps] if v is not None]
        if len(provided) != 1:
            raise ValueError("Provide exactly one throughput value (Mbps, kbps, or bps)")
        return self

    def target_bps(self) -> int:
        if self.Mbps is not None:
            return int(self.Mbps * 1_000_000)
        if self.kbps is not None:
            return int(self.kbps * 1_000)
        return int(self.bps or 0)


# Response Schemas
class StatusResponse(BaseModel):
    status: str


class AttachResponse(StatusResponse):
    ue_id: int


class DetachResponse(StatusResponse):
    ue_id: int


class BearerAddResponse(StatusResponse):
    ue_id: int
    bearer_id: int


class BearerDeleteResponse(StatusResponse):
    ue_id: int
    bearer_id: int


class TrafficStartResponse(StatusResponse):
    ue_id: int
    bearer_id: int
    target_bps: int


class TrafficStopResponse(StatusResponse):
    ue_id: int
    bearer_id: int


class TrafficStatsResponse(BaseModel):
    ue_id: int
    bearer_id: int
    protocol: str | None = None
    target_bps: int | None = None
    tx_bps: int
    rx_bps: int
    duration: float


class UEDisplayResponse(UEState):
    pass


class UEListResponse(BaseModel):
    ues: list[int]


class AggregatedStatsResponse(BaseModel):
    scope: str  # 'all' or f'ue:{id}'
    ue_count: int
    bearer_count: int
    total_tx_bps: int
    total_rx_bps: int
    details: dict[str, dict[str, int]] | None = None  # per ue optional


db.py


In [15]:
"""SQLite-backed EPC repository."""

import os
import sqlite3

#from .models import BearerConfig, ThroughputStats, UEState

# Default path: file in cwd. Tests should use a dedicated tempfile.
EPC_DB_PATH = os.getenv("EPC_DB_PATH", "epc.db")

_SCHEMA = """
CREATE TABLE IF NOT EXISTS ue_state (
    ue_id INTEGER PRIMARY KEY,
    data TEXT NOT NULL
);
"""


class EPCRepository:
    """Repository for UE state and bearers, backed by SQLite (file path)."""

    def __init__(self, db_path: str | None = None):
        self._path = db_path if db_path is not None else EPC_DB_PATH
        self._init_schema()

    def _conn(self) -> sqlite3.Connection:
        conn = sqlite3.connect(self._path)
        conn.row_factory = sqlite3.Row
        return conn

    def _init_schema(self) -> None:
        with self._conn() as c:
            c.executescript(_SCHEMA)

    def list_ues(self):
        with self._conn() as c:
            for row in c.execute("SELECT ue_id FROM ue_state ORDER BY ue_id"):
                yield int(row["ue_id"])

    def ue_exists(self, ue_id: int) -> bool:
        with self._conn() as c:
            cur = c.execute("SELECT 1 FROM ue_state WHERE ue_id = ?", (ue_id,))
            return cur.fetchone() is not None

    def attach_ue(self, ue_id: int) -> None:
        if self.ue_exists(ue_id):
            raise ValueError("UE already attached")
        state = UEState(ue_id=ue_id)
        state.bearers[9] = BearerConfig(bearer_id=9)
        with self._conn() as c:
            c.execute(
                "INSERT INTO ue_state (ue_id, data) VALUES (?, ?)",
                (ue_id, state.model_dump_json()),
            )

    def detach_ue(self, ue_id: int) -> None:
        if not self.ue_exists(ue_id):
            raise ValueError("UE not found")
        with self._conn() as c:
            c.execute("DELETE FROM ue_state WHERE ue_id = ?", (ue_id,))

    def get_ue(self, ue_id: int) -> UEState:
        with self._conn() as c:
            cur = c.execute("SELECT data FROM ue_state WHERE ue_id = ?", (ue_id,))
            row = cur.fetchone()
        if not row:
            raise ValueError("UE not found")
        raw = row["data"]
        return UEState.model_validate_json(raw)

    def save_ue(self, state: UEState) -> None:
        with self._conn() as c:
            c.execute(
                "INSERT OR REPLACE INTO ue_state (ue_id, data) VALUES (?, ?)",
                (state.ue_id, state.model_dump_json()),
            )

    def add_bearer(self, ue_id: int, bearer_id: int) -> None:
        state = self.get_ue(ue_id)
        if bearer_id in state.bearers:
            raise ValueError("Bearer already exists")
        state.bearers[bearer_id] = BearerConfig(bearer_id=bearer_id)
        self.save_ue(state)

    def update_bearer(self, ue_id: int, bearer: BearerConfig) -> None:
        state = self.get_ue(ue_id)
        state.bearers[bearer.bearer_id] = bearer
        self.save_ue(state)

    def update_stats(self, ue_id: int, stats: ThroughputStats) -> None:
        state = self.get_ue(ue_id)
        state.stats[stats.bearer_id] = stats
        self.save_ue(state)

    def reset_all(self) -> None:
        for ue_id in list(self.list_ues()):
            self.detach_ue(ue_id)

    def delete_bearer(self, ue_id: int, bearer_id: int) -> None:
        if bearer_id == 9:
            raise ValueError("Cannot remove default bearer")
        state = self.get_ue(ue_id)
        if bearer_id not in state.bearers:
            raise ValueError("Bearer not found")
        state.bearers.pop(bearer_id, None)
        state.stats.pop(bearer_id, None)
        self.save_ue(state)


traffic.py


In [16]:
import asyncio
import threading
import time
from concurrent.futures import Future

#from .models import BearerConfig, ThroughputStats
#from .db import EPCRepository

# Dedicated background asyncio loop for traffic tasks (works inside test threadpool)
_traffic_loop = asyncio.new_event_loop()


def _run_background_loop(loop: asyncio.AbstractEventLoop):
    asyncio.set_event_loop(loop)
    loop.run_forever()


_traffic_thread = threading.Thread(target=_run_background_loop, args=(_traffic_loop,), daemon=True)
_traffic_thread.start()


class TrafficGeneratorManager:
    def __init__(self, repo: EPCRepository):
        self.repo = repo
        self.tasks: dict[tuple[int, int], Future] = {}

    async def _run_simulated_bearer(self, ue_id: int, bearer_id: int, target_bps: int, protocol: str):
        interval = 1.0  # seconds per update
        bytes_per_interval = int(target_bps / 8 * interval)  # convert bps to bytes/sec
        while True:
            state = self.repo.get_ue(ue_id)
            stats = state.stats.get(bearer_id)
            if not stats:
                stats = ThroughputStats(bearer_id=bearer_id, ue_id=ue_id, start_ts=time.time())
            if stats.start_ts is None:
                stats.start_ts = time.time()
            stats.last_update_ts = time.time()
            stats.bytes_tx += bytes_per_interval
            stats.bytes_rx += bytes_per_interval
            stats.protocol = protocol
            stats.target_bps = target_bps
            self.repo.update_stats(ue_id, stats)
            await asyncio.sleep(interval)

    def start(self, ue_id: int, bearer: BearerConfig):
        key = (ue_id, bearer.bearer_id)
        if key in self.tasks:
            raise ValueError("Traffic already running")
        if not bearer.target_bps or not bearer.protocol:
            raise ValueError("Bearer not configured for traffic")
        future = asyncio.run_coroutine_threadsafe(
            self._run_simulated_bearer(ue_id, bearer.bearer_id, bearer.target_bps, bearer.protocol),
            _traffic_loop,
        )
        self.tasks[key] = future

    def stop(self, ue_id: int, bearer_id: int):
        key = (ue_id, bearer_id)
        future = self.tasks.get(key)
        if future:
            future.cancel()
            del self.tasks[key]

    def stop_all(self):
        for key, future in list(self.tasks.items()):
            future.cancel()
            del self.tasks[key]

    def is_running(self, ue_id: int, bearer_id: int) -> bool:
        return (ue_id, bearer_id) in self.tasks


traffic_manager: TrafficGeneratorManager | None = None


def get_traffic_manager(repo: EPCRepository) -> TrafficGeneratorManager:
    global traffic_manager
    if traffic_manager is None:
        traffic_manager = TrafficGeneratorManager(repo)
    return traffic_manager


api.py

In [25]:
pip install "fastapi"

Note: you may need to restart the kernel to use updated packages.


In [26]:
import time
from typing import Annotated

from fastapi import APIRouter, Depends, HTTPException


#from .db import EPCRepository
#from .traffic import get_traffic_manager

router = APIRouter()

_repo_singleton: EPCRepository | None = None


def get_repo() -> EPCRepository:
    global _repo_singleton
    if _repo_singleton is None:
        _repo_singleton = EPCRepository()
    return _repo_singleton


@router.get("/ues/stats", response_model=AggregatedStatsResponse)
def get_ues_stats(
    repo: Annotated[EPCRepository, Depends(get_repo)],
    ue_id: int | None = None,
    include_details: bool = False,
):
    if ue_id is not None and not repo.ue_exists(ue_id):
        raise HTTPException(status_code=400, detail="UE not found")
    ues = [ue_id] if ue_id is not None else list(repo.list_ues())
    total_tx = 0
    total_rx = 0
    bearer_count = 0
    details: dict[str, dict[str, int]] = {}
    tm = get_traffic_manager(repo)
    for uid in ues:
        try:
            state = repo.get_ue(uid)
        except ValueError:
            if ue_id is not None:
                raise HTTPException(status_code=400, detail="UE not found")
            continue
        for b_id, stats in state.stats.items():
            end_ts = time.time() if (stats.start_ts and tm.is_running(uid, b_id)) else stats.last_update_ts
            duration = (end_ts - stats.start_ts) if (stats.start_ts and end_ts is not None) else 0
            tx_bps = int(stats.bytes_tx * 8 / duration) if duration > 0 else 0
            rx_bps = int(stats.bytes_rx * 8 / duration) if duration > 0 else 0
            total_tx += tx_bps
            total_rx += rx_bps
            bearer_count += 1
            if include_details:
                details.setdefault(str(uid), {})[str(b_id)] = tx_bps
    scope = f"ue:{ue_id}" if ue_id is not None else "all"
    return AggregatedStatsResponse(
        scope=scope,
        ue_count=len(ues),
        bearer_count=bearer_count,
        total_tx_bps=total_tx,
        total_rx_bps=total_rx,
        details=details if include_details else None,
    )


@router.get("/ues", response_model=UEListResponse)
def list_ues(repo: Annotated[EPCRepository, Depends(get_repo)]):
    return UEListResponse(ues=list(repo.list_ues()))


@router.post("/ues", response_model=AttachResponse)
def attach_ue(body: AttachUERequest, repo: Annotated[EPCRepository, Depends(get_repo)]):
    try:
        repo.attach_ue(body.ue_id)
    except ValueError as e:
        raise HTTPException(status_code=400, detail=str(e))
    return AttachResponse(status="attached", ue_id=body.ue_id)


@router.get("/ues/{ue_id}", response_model=UEDisplayResponse)
def get_ue(ue_id: int, repo: Annotated[EPCRepository, Depends(get_repo)]):
    try:
        state = repo.get_ue(ue_id)
    except ValueError as e:
        raise HTTPException(status_code=400, detail=str(e))
    return UEDisplayResponse(**state.model_dump())


@router.delete("/ues/{ue_id}", response_model=DetachResponse)
def detach_ue(ue_id: int, repo: Annotated[EPCRepository, Depends(get_repo)]):
    try:
        repo.detach_ue(ue_id)
    except ValueError as e:
        raise HTTPException(status_code=400, detail=str(e))
    return DetachResponse(status="detached", ue_id=ue_id)


# --- Bearers ---

@router.post("/ues/{ue_id}/bearers", response_model=BearerAddResponse)
def add_bearer(
    ue_id: int,
    body: AddBearerRequest,
    repo: Annotated[EPCRepository, Depends(get_repo)],
):
    try:
        repo.add_bearer(ue_id, body.bearer_id)
    except ValueError as e:
        raise HTTPException(status_code=400, detail=str(e))
    return BearerAddResponse(status="bearer_added", ue_id=ue_id, bearer_id=body.bearer_id)


@router.delete("/ues/{ue_id}/bearers/{bearer_id}", response_model=BearerDeleteResponse)
def delete_bearer(
    ue_id: int,
    bearer_id: int,
    repo: Annotated[EPCRepository, Depends(get_repo)],
):
    try:
        state = repo.get_ue(ue_id)
    except ValueError as e:
        raise HTTPException(status_code=400, detail=str(e))
    if bearer_id not in state.bearers:
        raise HTTPException(status_code=400, detail="Bearer not found")
    tm = get_traffic_manager(repo)
    if tm.is_running(ue_id, bearer_id):
        tm.stop(ue_id, bearer_id)
    try:
        repo.delete_bearer(ue_id, bearer_id)
    except ValueError as e:
        raise HTTPException(status_code=400, detail=str(e))
    return BearerDeleteResponse(status="bearer_deleted", ue_id=ue_id, bearer_id=bearer_id)


# --- Traffic (start/stop/stats) ---

@router.post("/ues/{ue_id}/bearers/{bearer_id}/traffic", response_model=TrafficStartResponse)
def start_traffic(
    ue_id: int,
    bearer_id: int,
    body: StartTrafficRequest,
    repo: Annotated[EPCRepository, Depends(get_repo)],
):
    target_bps = body.target_bps()
    try:
        state = repo.get_ue(ue_id)
    except ValueError as e:
        raise HTTPException(status_code=400, detail=str(e))
    bearer = state.bearers.get(bearer_id)
    if not bearer:
        raise HTTPException(status_code=400, detail="Bearer not found")
    bearer.protocol = body.protocol.lower()
    bearer.target_bps = target_bps
    bearer.active = True
    repo.update_bearer(ue_id, bearer)
    from .models import ThroughputStats

    if bearer_id not in state.stats:
        initial_stats = ThroughputStats(
            bearer_id=bearer_id,
            ue_id=ue_id,
            start_ts=time.time(),
            last_update_ts=time.time(),
            protocol=bearer.protocol,
            target_bps=target_bps,
        )
        repo.update_stats(ue_id, initial_stats)
    tm = get_traffic_manager(repo)
    try:
        tm.start(ue_id, bearer)
    except ValueError as e:
        raise HTTPException(status_code=400, detail=str(e))
    return TrafficStartResponse(
        status="traffic_started",
        ue_id=ue_id,
        bearer_id=bearer_id,
        target_bps=target_bps,
    )


@router.delete("/ues/{ue_id}/bearers/{bearer_id}/traffic", response_model=TrafficStopResponse)
def stop_traffic(
    ue_id: int,
    bearer_id: int,
    repo: Annotated[EPCRepository, Depends(get_repo)],
):
    try:
        state = repo.get_ue(ue_id)
    except ValueError as e:
        raise HTTPException(status_code=400, detail=str(e))
    bearer = state.bearers.get(bearer_id)
    if not bearer:
        raise HTTPException(status_code=400, detail="Bearer not found")
    tm = get_traffic_manager(repo)
    tm.stop(ue_id, bearer_id)
    bearer.active = False
    repo.update_bearer(ue_id, bearer)
    return TrafficStopResponse(status="traffic_stopped", ue_id=ue_id, bearer_id=bearer_id)


@router.get("/ues/{ue_id}/bearers/{bearer_id}/traffic", response_model=TrafficStatsResponse)
def get_traffic_stats(
    ue_id: int,
    bearer_id: int,
    repo: Annotated[EPCRepository, Depends(get_repo)],
):
    try:
        state = repo.get_ue(ue_id)
    except ValueError as e:
        raise HTTPException(status_code=400, detail=str(e))
    stats = state.stats.get(bearer_id)
    if not stats:
        return TrafficStatsResponse(
            ue_id=ue_id,
            bearer_id=bearer_id,
            protocol=None,
            target_bps=None,
            tx_bps=0,
            rx_bps=0,
            duration=0,
        )
    tm = get_traffic_manager(repo)
    end_ts = time.time() if (stats.start_ts and tm.is_running(ue_id, bearer_id)) else stats.last_update_ts
    duration = (end_ts - stats.start_ts) if (stats.start_ts and end_ts is not None) else 0
    tx_bps = int(stats.bytes_tx * 8 / duration) if duration > 0 else 0
    rx_bps = int(stats.bytes_rx * 8 / duration) if duration > 0 else 0
    return TrafficStatsResponse(
        ue_id=ue_id,
        bearer_id=bearer_id,
        protocol=stats.protocol,
        target_bps=stats.target_bps,
        tx_bps=tx_bps,
        rx_bps=rx_bps,
        duration=duration,
    )


# --- Reset ---

@router.post("/reset", response_model=StatusResponse)
def reset_all(repo: Annotated[EPCRepository, Depends(get_repo)]):
    get_traffic_manager(repo).stop_all()
    repo.reset_all()
    return StatusResponse(status="reset")


TypeError: GenerateSchema.__init__() got an unexpected keyword argument 'ns_resolver'

# TESTY 5

## Test 1 Stop Transfer On Active Bearer

In [27]:
import unittest
import time
from fastapi.testclient import TestClient



class TestStopTransfer(unittest.TestCase):
    
    def setUp(self):
        """Przygotowanie środowiska przed każdym testem (odpowiednik Prepare Environment)"""
        self.client = client # używamy klienta FastAPI zdefiniowanego wcześniej
        self.client.post("/reset")
        self.ue_id = 10
        self.bearer_id = 1

    def test_tc_5_01_stop_transfer_on_active_bearer(self):
        """TC_5 01: Test if possible to stop active bearer"""
        
        resp = self.client.post("/ues", json={"ue_id": self.ue_id})
        self.assertEqual(resp.status_code, 200)
    
        resp = self.client.post(f"/ues/{self.ue_id}/bearers", json={"bearer_id": self.bearer_id})
        self.assertEqual(resp.status_code, 200)
       
        resp = self.client.post(
            f"/ues/{self.ue_id}/bearers/{self.bearer_id}/traffic", 
            json={"protocol": "udp", "Mbps": 50}
        )
        self.assertEqual(resp.status_code, 200)
        
        time.sleep(1.1)
        
        resp = self.client.get(f"/ues/{self.ue_id}/bearers/{self.bearer_id}/traffic")
        stats_before = resp.json()
        tx_before = stats_before.get("tx_bps", 0)
        print(f"\n[DEBUG] Tx bps przed stopem: {tx_before}")
        
        self.assertGreater(tx_before, 0, "Przed stopem ruch (tx_bps) powinien być większy niż 0")
        
        resp = self.client.delete(f"/ues/{self.ue_id}/bearers/{self.bearer_id}/traffic")
        self.assertEqual(resp.status_code, 200)
        
        
        resp = self.client.get(f"/ues/{self.ue_id}/bearers/{self.bearer_id}/traffic")
       
        self.assertEqual(resp.json().get("status", "stopped"), "stopped")


if __name__ == '__main__':
    unittest.main(argv=['first-arg-is-ignored'], exit=False)

E
ERROR: test_tc_5_01_stop_transfer_on_active_bearer (__main__.TestStopTransfer.test_tc_5_01_stop_transfer_on_active_bearer)
TC_5 01: Test if possible to stop active bearer
----------------------------------------------------------------------
Traceback (most recent call last):
  File "C:\Users\Nikodem\AppData\Local\Temp\ipykernel_32976\2836995290.py", line 11, in setUp
    self.client = client # używamy klienta FastAPI zdefiniowanego wcześniej
                  ^^^^^^
NameError: name 'client' is not defined

----------------------------------------------------------------------
Ran 1 test in 0.003s

FAILED (errors=1)
